In [21]:
import pandas as pd
import numpy as np
import h5py 
from sklearn.preprocessing import StandardScaler
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import STL


In [ ]:
import h5py
import pandas as pd

with h5py.File('METR-LA.h5', 'r') as f:

    keys = list(f.keys())
    print("H5 Keys:", keys)

    
    if 'df' in keys:
        
        data = f['df']['block0_values'][:]
        axis0 = [x.decode('utf-8') if isinstance(x, bytes) else x for x in f['df']['axis0'][:]]  
        axis1 = f['df']['axis1'][:]  
        
        # DataFrame oluşturuluyor   
        df = pd.DataFrame(data, index=pd.to_datetime(axis1), columns=axis0)
    else:
        # Eğer direkt kök dizinde dataset olarak duruyorsa:
        dataset_name = keys[0]
        data = f[dataset_name][:]
        df = pd.DataFrame(data)

print(df.shape)
print(df.head())

H5 İçeriği: ['df']
(34272, 207)
                        773869     767541     767542     717447     717446  \
2012-03-01 00:00:00  64.375000  67.625000  67.125000  61.500000  66.875000   
2012-03-01 00:05:00  62.666667  68.555556  65.444444  62.444444  64.444444   
2012-03-01 00:10:00  64.000000  63.750000  60.000000  59.000000  66.500000   
2012-03-01 00:15:00   0.000000   0.000000   0.000000   0.000000   0.000000   
2012-03-01 00:20:00   0.000000   0.000000   0.000000   0.000000   0.000000   

                        717445  773062  767620     737529     717816  ...  \
2012-03-01 00:00:00  68.750000  65.125  67.125  59.625000  62.750000  ...   
2012-03-01 00:05:00  68.111111  65.000  65.000  57.444444  63.333333  ...   
2012-03-01 00:10:00  66.250000  64.500  64.250  63.875000  65.375000  ...   
2012-03-01 00:15:00   0.000000   0.000   0.000   0.000000   0.000000  ...   
2012-03-01 00:20:00   0.000000   0.000   0.000   0.000000   0.000000  ...   

                        772167  7693

In [30]:
null_count = df.isnull().sum().sum()
zero_count = (df == 0.0).sum().sum()
total_cells = df.size

print("Eksik Değer Sayısı:", null_count)
print("Sıfır Değer Sayısı:", zero_count)
print("Toplam Hücre Sayısı:", total_cells)

Eksik Değer Sayısı: 0
Sıfır Değer Sayısı: 575302
Toplam Hücre Sayısı: 7094304


In [ ]:
df=df.replace(0.0, np.nan) #interpolasyon öncesi sıfırları NaN ile değiştiriyoruz


In [ ]:
df=df.interpolate(method='linear', limit_direction='both', axis=0) #interpolasyon işlemi ile eksik değerleri dolduruyoruz


In [ ]:
from sklearn.model_selection import TimeSeriesSplit 

time_series_split = TimeSeriesSplit(n_splits=5)

fold= 1
for train_index, test_index in time_series_split.split(df):
    print(f"Fold {fold}:")
    train_df= df.iloc[train_index]
    test_df= df.iloc[test_index]
   

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_df)
    X_test_scaled = scaler.transform(test_df)
    print(f" Normalize Edilmiş Değerlerin Eğitim seti sayısı: {X_train_scaled.shape[0]}, Test sayısı: {X_test_scaled.shape[0]}")
    fold += 1
print(X_train_scaled.shape) #Normalizasyon sonrası eğitim seti boyutu kontrolü
print(X_test_scaled.shape)#Normalizasyon sonrası test seti boyutu kontrolü

Fold 1:
 Normalize Edilmiş Değerlerin Eğitim seti sayısı: 5712, Test sayısı: 5712
Fold 2:
 Normalize Edilmiş Değerlerin Eğitim seti sayısı: 11424, Test sayısı: 5712
Fold 3:
 Normalize Edilmiş Değerlerin Eğitim seti sayısı: 17136, Test sayısı: 5712
Fold 4:
 Normalize Edilmiş Değerlerin Eğitim seti sayısı: 22848, Test sayısı: 5712
Fold 5:
 Normalize Edilmiş Değerlerin Eğitim seti sayısı: 28560, Test sayısı: 5712
(28560, 207)
(5712, 207)
